### Minimal-change objective

Our goal is not to regenerate the removed semantic part, but to repair the damaged mesh locally around the removal region. In particular, the repair should close the target openings caused by part removal while keeping the modification as local as possible. Ideally, changes should be concentrated inside a repair zone near the removed part, with minimal geometric or topological changes to unaffected regions.

Under this formulation, a good repair method should satisfy two requirements:

1. **Opening closure**: the target boundary loops created by part removal should be eliminated or significantly reduced.
2. **Locality / minimal change**: the introduced patch geometry should remain near the removed-part region, without unnecessarily affecting distant parts of the mesh.

### Evaluation protocol

We evaluate repair quality from two aspects: opening closure and locality.

**1. Opening closure**

We first extract all boundary loops from the damaged mesh `M_d`.  
Then, using the removed-part mesh, we compute its bounding box and select the boundary loops near this region as the target loops to repair.

For these target loops, we measure:

- **total loop length before repair**: the sum of target-loop perimeters on `M_d`
- **target loop length after repair**: the sum of target-loop perimeters near the same region on `M_r`
- **improvement**: the difference between the two

A better repair method should reduce the target-loop length after repair as much as possible.

**2. Locality / minimal change**

To evaluate whether the repair remains local, we define a repair zone using the removed-part bounding box with a small margin.  
We then examine the newly added patch faces in `M_r` and classify each added face by whether its face center lies inside or outside the repair zone.

We report:

- number of added faces inside the repair zone
- number of added faces outside the repair zone
- face locality ratio

A better repair method should keep most added faces inside the repair zone and avoid unnecessary changes outside it.

### Experimental setup

We conduct experiments on a 50-sample Chair subset constructed from PartNet data_v0. For each sample, one semantic leg part is removed from the original mesh to produce a damaged mesh. The dataset is split into 35 training samples, 5 validation samples, and 10 test samples.

We evaluate two geometric repair baselines:

- **Center-fan baseline**: each target boundary loop is closed by inserting one center vertex and forming a fan of triangles.
- **Planar triangulation baseline**: each target boundary loop is projected onto a locally fitted plane and triangulated without introducing new vertices.

All quantitative results reported here are computed on the test split.

### Baseline comparison

Both baselines can close the target openings on the test set, achieving the same mean target-loop reduction:

- mean total loop length before repair: 0.6681
- mean target loop length after repair: 0.0000
- mean improvement: 0.6681

However, their patch characteristics are different:

- **Center-fan baseline**
  - mean number of new vertices: 1.8
  - mean number of added faces: 52.11
  - mean added-face quality: 0.4457
  - mean minimum added-face quality: 0.2112

- **Planar triangulation baseline**
  - mean number of new vertices: 0.0
  - mean number of added faces: 41.11
  - mean added-face quality: 0.4574
  - mean minimum added-face quality: 0.2487

Therefore, although both methods close the openings equally well under the current perimeter-based metric, the planar triangulation baseline produces cleaner patches with fewer added faces and better triangle quality overall.

### Removed-part-aware target selection vs. naive largest-hole repair

To verify that target-loop selection is a key component of this task, we further compare two variants under the same planar triangulation repair method:

Removed-part-aware planar repair: target loops are selected using the removed-part bounding box. Naive largest-hole planar repair: only the largest boundary loop is selected, without using removed-part information.

This comparison isolates the effect of target selection, while keeping the repair geometry generation procedure unchanged.

On the test set, the removed-part-aware planar baseline achieves:

mean target loop length after repair: 0.0000 mean improvement: 0.6681

In contrast, the naive largest-hole baseline achieves:

mean target loop length after repair: 0.5729 mean improvement: 0.7168

Although the naive largest-hole baseline can reduce boundary length in some cases, it often fails to eliminate the target openings near the removed-part region. In several test samples, a substantial amount of target-loop perimeter remains after repair, indicating that the method repairs a different hole rather than the one caused by semantic part removal.

This result shows that, for the proposed task, generic largest-hole repair is not sufficient. The key difficulty is not only how to triangulate a boundary loop, but also how to identify the correct opening associated with the removed semantic part. Therefore, removed-part-aware target selection is an essential component of minimal-change local mesh repair after semantic part removal.

### Locality analysis

To better evaluate the minimal-change property, we additionally measure the locality of the newly added patch faces. Specifically, we define a repair zone by the removed-part bounding box with a small margin, and classify each added face according to whether its face center lies inside or outside this zone.

On the test set, both baselines remain largely local under this metric:

- **Center-fan baseline**
  - mean added faces inside zone: 32.3
  - mean added faces outside zone: 14.6
  - mean face locality ratio: 0.6786

- **Planar triangulation baseline**
  - mean added faces inside zone: 28.8
  - mean added faces outside zone: 8.2
  - mean face locality ratio: 0.6744

Overall, the two baselines show very similar locality ratios. However, the planar triangulation baseline uses fewer added faces in total and introduces fewer faces outside the repair zone, suggesting a slightly cleaner local repair behavior.

### Overall takeaway

The current experiments show that both baselines can reliably close the target openings caused by semantic part removal. Under the perimeter-based closure metric, they achieve the same repair effectiveness on the test set.

However, the planar triangulation baseline is preferable overall. It closes the openings without introducing new vertices, uses fewer added faces, and produces better triangle quality. Under the locality analysis, both methods remain largely local, while planar triangulation introduces fewer faces outside the repair zone. Therefore, among the current geometric baselines, planar triangulation provides a cleaner and more minimal local repair behavior.